In [ ]:

import pyodbc
from ollama import chat


# ============================================================
# 1. SQL SERVER CONNECTION
# ============================================================

connection = pyodbc.connect(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=DESKTOP-GC4P189;"
    "DATABASE=orderssystem;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

cursor = connection.cursor()


# ============================================================
# 2. SALESMAN PERFORMANCE AGENT
# ============================================================

def salesman_performance_agent(user_prompt):

    # --------------------------------------------------------
    # QUERY 1: OVERALL SALESMAN PERFORMANCE
    # --------------------------------------------------------

    cursor.execute("""
        SELECT
            s.salesmanid,
            s.salesmanname,
            COUNT(o.orderid) AS total_orders,
            COALESCE(SUM(o.amount), 0) AS total_revenue,
            COALESCE(AVG(o.amount), 0) AS average_order_value,
            COUNT(DISTINCT o.custid) AS unique_customers,
            COALESCE(MIN(o.amount), 0) AS minimum_order,
            COALESCE(MAX(o.amount), 0) AS maximum_order

        FROM salesman s

        LEFT JOIN orders o
            ON s.salesmanid = o.salesmanid

        GROUP BY
            s.salesmanid,
            s.salesmanname

        ORDER BY total_revenue DESC
    """)

    rows = cursor.fetchall()

    performance_data = "\n".join(
        f"Salesman ID: {r.salesmanid}, "
        f"Salesman: {r.salesmanname}, "
        f"Total Orders: {r.total_orders}, "
        f"Total Revenue: {r.total_revenue}, "
        f"Average Order Value: {r.average_order_value}, "
        f"Unique Customers: {r.unique_customers}, "
        f"Minimum Order: {r.minimum_order}, "
        f"Maximum Order: {r.maximum_order}"
        for r in rows
    )


    # --------------------------------------------------------
    # QUERY 2: MONTH-WISE SALESMAN PERFORMANCE
    # --------------------------------------------------------

    cursor.execute("""
        SELECT
            s.salesmanid,
            s.salesmanname,
            YEAR(o.dateofpurchase) AS sales_year,
            MONTH(o.dateofpurchase) AS sales_month,
            COUNT(o.orderid) AS total_orders,
            SUM(o.amount) AS total_revenue

        FROM orders o

        INNER JOIN salesman s
            ON o.salesmanid = s.salesmanid

        GROUP BY
            s.salesmanid,
            s.salesmanname,
            YEAR(o.dateofpurchase),
            MONTH(o.dateofpurchase)

        ORDER BY
            sales_year,
            sales_month,
            total_revenue DESC
    """)

    rows = cursor.fetchall()

    monthly_data = "\n".join(
        f"Salesman: {r.salesmanname}, "
        f"Year: {r.sales_year}, "
        f"Month: {r.sales_month}, "
        f"Orders: {r.total_orders}, "
        f"Revenue: {r.total_revenue}"
        for r in rows
    )


    # --------------------------------------------------------
    # QUERY 3: CUSTOMER-WISE SALESMAN PERFORMANCE
    # --------------------------------------------------------

    cursor.execute("""
        SELECT
            s.salesmanid,
            s.salesmanname,
            c.custid,
            c.custname,
            COUNT(o.orderid) AS total_orders,
            SUM(o.amount) AS total_revenue

        FROM orders o

        INNER JOIN salesman s
            ON o.salesmanid = s.salesmanid

        INNER JOIN customer c
            ON o.custid = c.custid

        GROUP BY
            s.salesmanid,
            s.salesmanname,
            c.custid,
            c.custname

        ORDER BY
            s.salesmanname,
            total_revenue DESC
    """)

    rows = cursor.fetchall()

    customer_data = "\n".join(
        f"Salesman: {r.salesmanname}, "
        f"Customer: {r.custname}, "
        f"Orders: {r.total_orders}, "
        f"Revenue: {r.total_revenue}"
        for r in rows
    )


    # --------------------------------------------------------
    # QUERY 4: SALESMAN REVENUE CONTRIBUTION
    # --------------------------------------------------------

    cursor.execute("""
        SELECT
            s.salesmanid,
            s.salesmanname,
            SUM(o.amount) AS total_revenue,

            CAST(
                SUM(o.amount) * 100.0 /
                NULLIF(
                    (SELECT SUM(amount) FROM orders),
                    0
                )
                AS DECIMAL(10,2)
            ) AS revenue_contribution_percentage

        FROM orders o

        INNER JOIN salesman s
            ON o.salesmanid = s.salesmanid

        GROUP BY
            s.salesmanid,
            s.salesmanname

        ORDER BY total_revenue DESC
    """)

    rows = cursor.fetchall()

    contribution_data = "\n".join(
        f"Salesman: {r.salesmanname}, "
        f"Revenue: {r.total_revenue}, "
        f"Revenue Contribution: "
        f"{r.revenue_contribution_percentage}%"
        for r in rows
    )


    # --------------------------------------------------------
    # 3. SEND DATA TO OLLAMA
    # --------------------------------------------------------

    response = chat(
        model="llama3.2",
        messages=[
            {
                "role": "system",
                "content": """
You are an AI Salesman Performance Analysis Agent.

Your responsibility is to analyse salesman performance using
the SQL Server data supplied by the application.

RULES:

1. Use only the supplied database information.
2. Never invent salesmen, customers, orders, or figures.
3. Identify salesmen by name and ID whenever possible.
4. Compare revenue, orders, average order value, and customers.
5. Use the actual month and year in the data.
6. Explain revenue contribution percentages accurately.
7. Distinguish revenue from profit.
8. Do not claim that a salesman is profitable because revenue
   alone does not establish profit.
9. If information is unavailable, clearly state that.
10. Provide clear explanations and structured answers.
11. When comparing salesmen, show the relevant figures.
12. Give business recommendations only when supported by
    the available data.
"""
            },

            {
                "role": "user",
                "content": f"""
SALESMAN PERFORMANCE DATA:

{performance_data}


MONTH-WISE PERFORMANCE:

{monthly_data}


CUSTOMER-WISE PERFORMANCE:

{customer_data}


REVENUE CONTRIBUTION:

{contribution_data}


USER QUESTION:

{user_prompt}


Analyse the supplied data and answer the user's question
clearly and accurately.
"""
            }
        ]
    )

    return response.message.content


# ============================================================
# 4. USER INTERFACE
# ============================================================

print("\n==============================================")
print("       AI SALESMAN PERFORMANCE AGENT")
print("==============================================")

print("\nYou can ask ANY salesman-related question.")

print("\nExamples:")
print("  Who is the highest revenue-generating salesman?")
print("  Compare the performance of all salesmen.")
print("  Which salesman handled the most orders?")
print("  Who served the highest number of customers?")
print("  What is the average order value of each salesman?")
print("  Show the monthly performance of each salesman.")
print("  What percentage of revenue did each salesman generate?")
print("  Which salesman improved their revenue over time?")
print("  Give me a salesman performance report.")

print("\nType 'exit' to quit.")


# ============================================================
# 5. ACCEPT USER PROMPTS
# ============================================================

while True:

    user_prompt = input(
        "\nEnter your salesman performance question: "
    ).strip()

    if user_prompt.lower() == "exit":

        print("\nExiting Salesman Performance Agent...")
        break

    if not user_prompt:

        print("Please enter a question.")
        continue

    try:

        print("\nAnalysing salesman performance...")

        result = salesman_performance_agent(user_prompt)

        print("\n==============================================")
        print("       SALESMAN PERFORMANCE REPORT")
        print("==============================================")

        print(result)

        print("\n==============================================")

    except Exception as e:

        print("\nERROR:")
        print("----------------------------------------------")
        print(e)


# ============================================================
# 6. CLOSE DATABASE CONNECTION
# ============================================================

cursor.close()
connection.close()

print("\nDatabase connection closed.")
print("Salesman Performance Agent stopped.")


       AI SALESMAN PERFORMANCE AGENT

You can ask ANY salesman-related question.

Examples:
  Who is the highest revenue-generating salesman?
  Compare the performance of all salesmen.
  Which salesman handled the most orders?
  Who served the highest number of customers?
  What is the average order value of each salesman?
  Show the monthly performance of each salesman.
  What percentage of revenue did each salesman generate?
  Which salesman improved their revenue over time?
  Give me a salesman performance report.

Type 'exit' to quit.



Enter your salesman performance question:  Compare the performance of all salesmen



Analysing salesman performance...

       SALESMAN PERFORMANCE REPORT
To compare the performance of all salesmen, we need to analyze the data from various aspects. Here's a summary of the key findings:

**Revenue Comparison:**

The top three salesmen in terms of revenue are:

1. Manoj Ghosh: Revenue: 112,000.00
2. Sanjay Roy: Revenue: 104,500.00
3. Vikash Sen: Revenue: 91,000.00

These three salesmen contribute 59.73% (112,000 / 187,500) to the total revenue.

**Orders Comparison:**

The top three salesmen in terms of orders are:

1. Sanjay Roy: Orders: 5
2. Vikash Sen: Orders: 5
3. Manoj Ghosh: Orders: 4

These three salesmen have the highest number of orders, but their order volume is not necessarily correlated with revenue.

**Average Order Value (AOV) Comparison:**

The top three salesmen in terms of AOV are:

1. Manoj Ghosh: AOV: 28,000.00
2. Vikash Sen: AOV: 18,200.00
3. Sanjay Roy: AOV: 20,900.00

Manoj Ghosh has the highest AOV, which indicates that he is more effective at clo


Enter your salesman performance question:   What is the average order value of each salesman?



Analysing salesman performance...
